# Oyster Mushroom Cluster Monitoring

Run YOLO26-L-seg on your own mushroom images, track cluster masks across each sequence, review the predictions and growth histories, and export the results. The notebook works with unannotated images; no ground-truth files are needed.

## 1. Run it

1. Attach the [Oyster Mushroom Cluster Monitoring Models](https://www.kaggle.com/datasets/hnamedu/fungicirno-models) dataset to the Kaggle notebook.
2. Attach your image dataset and set `IMAGE_DIR` in the config cell below.
3. Run all cells.

The notebook looks for `selected_yolo26l_charisis.pt` in the attached model inputs. Set `YOLO_WEIGHTS_PATH` in the config cell only when you need to point to a different checkpoint. By default, it processes every discovered image.

Keep each camera view and each separate growing session in its own sequence. The tracker assigns IDs within a sequence; it should not compare images from different cameras or different growing sessions.

In [ ]:
from pathlib import Path

# =============================================================================
# CONFIG — change settings here only
# =============================================================================

# Input / output
IMAGE_DIR = "/kaggle/input/your-image-dataset"
YOLO_WEIGHTS_PATH = None  # None = find selected_yolo26l_charisis.pt in attached Kaggle inputs
OUTPUT_DIR = "/kaggle/working/monitoring_results"

# Image folders and ordering
SEQUENCE_MODE = "auto"          # auto | single | immediate_subfolders
ORDERING_MODE = "auto"          # auto | natural | datetime | custom
TIMESTAMP_REGEX = None           # custom regex; capture the timestamp in group 1
TIMESTAMP_FORMAT = None          # e.g. "%Y-%m-%d_%H%M"
FRAME_INTERVAL_MINUTES = None    # set only when the real capture interval is known
MAX_IMAGES_PER_SEQUENCE = None   # None = all images; set an integer only for a smaller test

# YOLO inference
YOLO_IMAGE_SIZE = 1024
YOLO_NMS_IOU = 0.70
YOLO_MAX_DETECTIONS = 1000
LOW_CONFIDENCE_FLOOR = 0.60
HIGH_CONFIDENCE_THRESHOLD = 0.8710562586784363

# Tracking
ASSOCIATION_IOU_THRESHOLD = 0.10
MAX_GAP = 0                      # missed sampled observations a track may bridge

# Measurements
GROWTH_DEADBAND_PERCENT = 5.0
REFERENCE_LENGTH_PX = None       # optional known reference length in pixels
REFERENCE_LENGTH_MM = None       # optional real length of that same reference

# Runtime and saved files
DEVICE = "auto"                  # auto | cpu | 0
EXPORT_OVERLAYS = True
EXPORT_BINARY_MASKS = True
CREATE_ZIP = True

# Console and inline review (these only limit what is displayed, not what is processed)
GROWTH_JUMP_WARNING_PERCENT = 30.0
CONSOLE_MAX_FRAME_ROWS = 60
CONSOLE_MAX_TRACK_ROWS = 50
CONSOLE_MAX_GROWTH_ROWS = 80
PREVIEW_MAX_IMAGES = 12
PREVIEW_MAX_TRACK_PLOTS = 12

## Image names and order

Tracking depends on processing each sequence in capture order. The default setting handles names such as `image_1.jpg`, `image_2.jpg`, and `image_10.jpg` in the expected numeric order. It also tries common date/time patterns in filenames.

For a different timestamp format, set `ORDERING_MODE = "custom"` in the config cell. The regular expression must capture the date/time as **group 1**. For example, for files named `camA_2026-05-01_0830.jpg`, use:

```python
TIMESTAMP_REGEX = r"(\d{4}-\d{2}-\d{2}_\d{4})"
TIMESTAMP_FORMAT = "%Y-%m-%d_%H%M"
```

If timestamps are not available, the notebook falls back to natural filename order. Set `FRAME_INTERVAL_MINUTES` only when you know the actual interval between images; it is used for elapsed-time and growth-rate calculations, not to fix incorrectly sorted filenames. Leave it as `None` when the interval is unknown.

**Folder layout:** with `SEQUENCE_MODE = "auto"`, image subfolders are treated as separate sequences. Explicitly named `camera`, `cam`, or `view` folders inside a session are split into separate streams too. A typical layout is:

```text
images/
  grow_session_1/
    camera_1/
      image_1.jpg
      image_2.jpg
    camera_2/
      image_1.jpg
      image_2.jpg
  grow_session_2/
    camera_1/
      image_1.jpg
      image_2.jpg
```

Use `SEQUENCE_MODE = "single"` only when the whole folder is one camera view from one continuous growing session. If your folder names do not identify cameras or sessions clearly, organize them into separate subfolders before running.

## 2. Load the model

The setup cells install/check the required packages and load the checkpoint linked above. If automatic checkpoint search cannot find the right file, set its full path in `YOLO_WEIGHTS_PATH` in the top config cell.

In [ ]:
import os, re, sys, csv, gc, json, time, math, shutil, hashlib, platform, subprocess, importlib.util
from datetime import datetime
from collections import defaultdict
from importlib.metadata import version, PackageNotFoundError

# Keep the runtime's existing Torch/CUDA build; install Ultralytics without replacing Torch.
REQUIRED_ULTRALYTICS = "8.4.214"
try:
    installed_ultralytics = version("ultralytics")
except PackageNotFoundError:
    installed_ultralytics = None
if installed_ultralytics != REQUIRED_ULTRALYTICS:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps",
        "--force-reinstall", f"ultralytics=={REQUIRED_ULTRALYTICS}"
    ])

missing = [name for name, module in [("opencv-python", "cv2"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("tqdm", "tqdm")] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import scipy
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm
from ultralytics import YOLO

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("Ultralytics:", version("ultralytics"), "| OpenCV:", cv2.__version__, "| SciPy:", scipy.__version__)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Check the image directory selected in the config cell.
SUPPORTED_IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
IMAGE_ROOT = Path(IMAGE_DIR).expanduser().resolve()
if not IMAGE_ROOT.is_dir():
    raise FileNotFoundError(f"IMAGE_DIR does not exist: {IMAGE_ROOT}")
if not any(p.is_file() and p.suffix.lower() in SUPPORTED_IMAGE_SUFFIXES for p in IMAGE_ROOT.rglob("*")):
    raise RuntimeError(f"No supported images found under {IMAGE_ROOT}")
print("Images:", IMAGE_ROOT)


In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_yolo_checkpoint(explicit_path=None):
    if explicit_path:
        path = Path(explicit_path).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"YOLO checkpoint does not exist: {path}")
        return path

    roots = [Path(p) for p in ["/kaggle/input", "/kaggle/working", Path.cwd(), "/mnt/data"] if Path(p).exists()]
    matches = set()
    for root in roots:
        try:
            matches.update(p.resolve() for p in root.rglob("selected_yolo26l_charisis.pt") if p.is_file())
        except (OSError, PermissionError):
            pass
    matches = sorted(matches, key=str)
    if not matches:
        raise FileNotFoundError(
            "Could not find selected_yolo26l_charisis.pt. Attach the Oyster Mushroom Cluster Monitoring Models "
            "Kaggle dataset or set YOLO_WEIGHTS_PATH to the checkpoint file."
        )
    if len(matches) > 1:
        hashes = {sha256_file(p) for p in matches}
        if len(hashes) > 1:
            raise RuntimeError("Multiple different YOLO checkpoints were found. Set YOLO_WEIGHTS_PATH explicitly: " + str(matches))
    return matches[0]


YOLO_CHECKPOINT = find_yolo_checkpoint(YOLO_WEIGHTS_PATH)
YOLO_CHECKPOINT_SHA256 = sha256_file(YOLO_CHECKPOINT)
yolo_model = YOLO(str(YOLO_CHECKPOINT))
yolo_names_raw = yolo_model.names
YOLO_CLASS_NAMES = ({int(k): str(v) for k, v in yolo_names_raw.items()}
                    if isinstance(yolo_names_raw, dict)
                    else {i: str(v) for i, v in enumerate(yolo_names_raw)})
print("Checkpoint:", YOLO_CHECKPOINT)
print("SHA-256:", YOLO_CHECKPOINT_SHA256)
print("Classes:", YOLO_CLASS_NAMES)

## 3. Prepare images for tracking

This section finds the supported image files, separates camera/session folders, and orders the images in each sequence. If your files use a custom timestamp format, set the parser options in the config cell before running. Check the printed sequence and ordering warnings before trusting the growth timeline.

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
AUTO_TIMESTAMP_PATTERNS = [
    (r"(\d{8}_\d{6})", "%Y%m%d_%H%M%S"),
    (r"(\d{8}_\d{4})", "%Y%m%d_%H%M"),
    (r"(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})", "%Y-%m-%dT%H:%M:%S"),
    (r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})", "%Y-%m-%d %H:%M:%S"),
    (r"(\d{4}-\d{2}-\d{2}T\d{2}-\d{2}-\d{2})", "%Y-%m-%dT%H-%M-%S"),
    (r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})", "%Y-%m-%d_%H-%M-%S"),
    (r"(\d{4}-\d{2}-\d{2}_\d{4})", "%Y-%m-%d_%H%M"),
    (r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2})", "%Y-%m-%d_%H-%M"),
]


def natural_key(value):
    # Type tags prevent int/string comparison errors for differently named files.
    return tuple((0, int(part)) if part.isdigit() else (1, part.casefold())
                 for part in re.split(r"(\d+)", str(value)))


def _visible_image_files(folder, output_root=None):
    folder = Path(folder)
    output_root = Path(output_root).resolve() if output_root else None
    paths = []
    for path in folder.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        if any(part.startswith(".") or part == "__MACOSX" for part in path.relative_to(folder).parts):
            continue
        resolved = path.resolve()
        if output_root is not None and (resolved == output_root or output_root in resolved.parents):
            continue
        paths.append(path)
    return sorted(paths, key=lambda p: natural_key(p.relative_to(folder).as_posix()))


def _looks_like_camera_folder(name):
    text = str(name).casefold().replace(" ", "_")
    return bool(re.search(r"(?:^|[_-])(?:camera|cam|view|viewpoint|perspective)(?:[_-]?[a-z0-9]+)?(?:$|[_-])", text))


def _camera_key_from_filename(path):
    # Common labels such as camera_1, cam2, view-03, or perspective_A.
    match = re.search(r"(?:^|[_.-])(?:camera|cam|view|viewpoint|perspective)[_.-]?([a-z0-9]+)(?=[_.-]|$)", path.stem, re.I)
    return match.group(1).casefold() if match else None


def discover_sequences(image_dir, sequence_mode="auto", output_dir=None):
    """Discover sequences and split explicit nested camera/view folders to avoid cross-view tracking."""
    root = Path(image_dir).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"IMAGE_DIR is not a directory: {root}")
    if sequence_mode not in {"auto", "single", "immediate_subfolders"}:
        raise ValueError("sequence_mode must be auto, single, or immediate_subfolders")

    if sequence_mode == "single":
        groups = [(root.name or "sequence", _visible_image_files(root, output_dir))]
    else:
        root_files = sorted(
            [p for p in root.iterdir() if p.is_file() and not p.name.startswith(".") and p.suffix.lower() in IMAGE_SUFFIXES],
            key=lambda p: natural_key(p.name)
        )
        child_groups = []
        children = sorted(
            [p for p in root.iterdir() if p.is_dir() and not p.name.startswith(".") and p.name != "__MACOSX"],
            key=lambda p: natural_key(p.name)
        )
        for child in children:
            # If a grow-tent folder contains multiple named camera/view folders,
            # keep each camera as its own independent chronological stream.
            nested_view_dirs = []
            for nested in sorted([p for p in child.iterdir() if p.is_dir() and not p.name.startswith(".")],
                                 key=lambda p: natural_key(p.name)):
                nested_files = _visible_image_files(nested, output_dir)
                if nested_files and _looks_like_camera_folder(nested.name):
                    nested_view_dirs.append((nested, nested_files))
            if len(nested_view_dirs) >= 2:
                child_groups.extend((f"{child.name}/{view_dir.name}", files)
                                    for view_dir, files in nested_view_dirs)
            else:
                files = _visible_image_files(child, output_dir)
                if files:
                    child_groups.append((child.name, files))

        if child_groups:
            groups = []
            if root_files:
                groups.append(("root_images", root_files))
            groups.extend(child_groups)
        else:
            # Flat directory: split only when camera/view IDs are explicit and unambiguous.
            flat_files = root_files or _visible_image_files(root, output_dir)
            camera_groups = {}
            for path in flat_files:
                camera_key = _camera_key_from_filename(path)
                if camera_key is not None:
                    camera_groups.setdefault(camera_key, []).append(path)
            camera_coverage = sum(map(len, camera_groups.values())) / max(1, len(flat_files))
            if len(camera_groups) >= 2 and camera_coverage >= 0.80:
                groups = [(f"camera_{key}", sorted(files, key=lambda p: natural_key(p.name)))
                          for key, files in sorted(camera_groups.items(), key=lambda x: natural_key(x[0]))]
                unassigned = [p for p in flat_files if _camera_key_from_filename(p) is None]
                if unassigned:
                    groups.append(("unassigned_images", sorted(unassigned, key=lambda p: natural_key(p.name))))
            else:
                groups = [(root.name or "sequence", flat_files)]

    sequences = []
    for index, (name, files) in enumerate(groups, start=1):
        if files:
            sequences.append({
                "sequence_id": f"seq_{index:03d}",
                "sequence_name": str(name),
                "sequence_dir": str(root),
                "files": files,
            })
    if not sequences:
        raise RuntimeError(f"No supported images were found under {root}. Supported extensions: {sorted(IMAGE_SUFFIXES)}")
    return sequences

def _parse_all_timestamps(files, regex, timestamp_format):
    pattern = re.compile(regex)
    parsed = {}
    matched = 0
    for path in files:
        match = pattern.search(path.name)
        if match is None:
            continue
        matched += 1
        try:
            parsed[path] = datetime.strptime(match.group(1), timestamp_format)
        except (ValueError, IndexError):
            continue
    if len(parsed) == len(files):
        return parsed, "complete"
    return parsed, "partial" if matched or parsed else "none"


def _auto_parse_timestamps(files):
    partial = False
    for regex, fmt in AUTO_TIMESTAMP_PATTERNS:
        parsed, state = _parse_all_timestamps(files, regex, fmt)
        if state == "complete":
            return parsed, regex, fmt, "complete"
        partial = partial or state == "partial"
    return None, None, None, "partial" if partial else "none"


def detect_source_frame_indices(files):
    """Extract a shared trailing frame number; tolerate isolated nonconforming filenames."""
    rx = re.compile(r"^(?P<prefix>.*?)(?P<number>\d+)(?P<suffix>\D*)$")
    parts = [rx.match(p.stem) for p in files]
    matched = [m for m in parts if m is not None]
    if not matched:
        return None
    prefixes = {m.group("prefix") for m in matched}
    suffixes = {m.group("suffix") for m in matched}
    indices = [int(m.group("number")) for m in matched]
    if len(prefixes) != 1 or len(suffixes) != 1 or len(set(indices)) != len(indices):
        return None
    prefix = next(iter(prefixes))
    # A trailing HHMM/SS token is not a frame index when the prefix already contains a date.
    if re.search(r"\d{4}[-_]\d{2}[-_]\d{2}", prefix):
        return None
    result = [int(m.group("number")) if m is not None else None for m in parts]
    known = [value for value in result if value is not None]
    if any(b <= a for a, b in zip(known, known[1:])):
        return None
    return result


def order_sequence(files, ordering_mode="auto", timestamp_regex=None, timestamp_format=None,
                   frame_interval_minutes=None):
    files = list(files)
    if ordering_mode not in {"auto", "natural", "datetime", "custom"}:
        raise ValueError("ordering_mode must be auto, natural, datetime, or custom")

    parsed = None
    parser_description = None
    parser_state = "not_requested"
    if ordering_mode in {"auto", "datetime"}:
        parsed, regex, fmt, parser_state = _auto_parse_timestamps(files)
        if parsed is not None:
            parser_description = {"mode": "auto_datetime", "regex": regex, "format": fmt}
    elif ordering_mode == "custom":
        if not timestamp_regex or not timestamp_format:
            raise ValueError("Custom ordering requires both TIMESTAMP_REGEX and TIMESTAMP_FORMAT")
        parsed, parser_state = _parse_all_timestamps(files, timestamp_regex, timestamp_format)
        if parsed:
            parser_description = {"mode": "custom", "regex": timestamp_regex, "format": timestamp_format}

    warnings_list = []
    if parsed is not None and len(parsed) == len(files):
        ordered = sorted(files, key=lambda p: (parsed[p], p.as_posix().casefold()))
        time_basis = "timestamp"
    else:
        ordered = sorted(files, key=lambda p: natural_key(p.as_posix()))
        if ordering_mode in {"auto", "datetime", "custom"}:
            reason = "some filenames did not parse" if parser_state == "partial" else "no complete timestamp pattern was found"
            warnings_list.append(f"Timestamp ordering unavailable ({reason}); natural sort used for the entire sequence.")
        parsed = None
        time_basis = "observation_index"

    source_indices = detect_source_frame_indices(ordered) if parsed is None else None
    if parsed is None and frame_interval_minutes is not None:
        known_indices = [value for value in (source_indices or []) if value is not None]
        if source_indices is not None and known_indices and all(b > a for a, b in zip(known_indices, known_indices[1:])):
            time_basis = "nominal_interval_from_filename_index"
        else:
            source_indices = None
            time_basis = "nominal_interval_from_observation_order"
    elif parsed is None:
        source_indices = None

    records = []
    first_timestamp = parsed[ordered[0]] if parsed else None
    known_source_indices = [value for value in (source_indices or []) if value is not None]
    first_source_index = known_source_indices[0] if known_source_indices else None
    for ordinal, path in enumerate(ordered):
        timestamp = parsed[path] if parsed else None
        source_index = source_indices[ordinal] if source_indices is not None else None
        if timestamp is not None:
            elapsed_minutes = (timestamp - first_timestamp).total_seconds() / 60.0
        elif frame_interval_minutes is not None:
            if source_index is not None and first_source_index is not None:
                frame_delta = source_index - first_source_index
                elapsed_minutes = float(frame_delta) * float(frame_interval_minutes)
            elif source_indices is None:
                elapsed_minutes = float(ordinal) * float(frame_interval_minutes)
            else:
                elapsed_minutes = None
        else:
            elapsed_minutes = None
        records.append({
            "path": path,
            "observation_order": ordinal,
            "frame_index": int(source_index) if source_index is not None else ordinal + 1,
            "source_frame_index": int(source_index) if source_index is not None else None,
            "timestamp": timestamp.isoformat(sep=" ") if timestamp is not None else None,
            "elapsed_minutes": float(elapsed_minutes) if elapsed_minutes is not None else None,
        })
    return records, {"time_basis": time_basis, "timestamp_parser": parser_description,
                     "timestamp_parse_state": parser_state, "warnings": warnings_list}

## 4. Segment and track clusters

YOLO produces one mask per detected cluster. The tracker matches masks between observations using mask IoU and one-to-one assignment.

- `HIGH_CONFIDENCE_THRESHOLD`: detections above this can start a new track.
- `LOW_CONFIDENCE_FLOOR`: lower-confidence detections are kept so they can recover an existing track, but they cannot start one.
- `ASSOCIATION_IOU_THRESHOLD`: minimum mask overlap for a match.
- `MAX_GAP`: number of missed observations a track may bridge. The default `0` closes a track as soon as an observation is missed.

These settings are all in the top config cell. Track IDs are local to each sequence.

In [ ]:
def mask_geometry(mask):
    mask = np.asarray(mask, dtype=bool)
    ys, xs = np.where(mask)
    if not len(xs):
        return None
    x0, x1 = int(xs.min()), int(xs.max()) + 1
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    return {
        "area_px": int(len(xs)),
        "bbox_x": x0, "bbox_y": y0,
        "bbox_width": x1 - x0, "bbox_height": y1 - y0,
        "centroid_x_px": float(xs.mean()), "centroid_y_px": float(ys.mean()),
    }


def _sync_for_device(device):
    if torch.cuda.is_available() and str(device).lower() not in {"cpu", "none"}:
        torch.cuda.synchronize()


def predict_yolo_image(model, image_bgr, config, device):
    """Default predictor adapter. Backend code can replace this callable."""
    height, width = image_bgr.shape[:2]
    _sync_for_device(device)
    started = time.perf_counter()
    result = model.predict(
        source=image_bgr,
        imgsz=int(config["YOLO_IMAGE_SIZE"]),
        conf=float(config["LOW_CONFIDENCE_FLOOR"]),
        iou=float(config["YOLO_NMS_IOU"]),
        max_det=int(config["YOLO_MAX_DETECTIONS"]),
        device=device,
        retina_masks=True,
        save=False,
        verbose=False,
    )[0]
    _sync_for_device(device)
    elapsed_seconds = time.perf_counter() - started

    detections = []
    if result.boxes is None or result.masks is None:
        return detections, elapsed_seconds
    scores = result.boxes.conf.detach().float().cpu().numpy()
    labels = result.boxes.cls.detach().int().cpu().numpy()
    masks = result.masks.data.detach().float().cpu().numpy()
    names_raw = getattr(model, "names", {})
    class_names = ({int(k): str(v) for k, v in names_raw.items()}
                   if isinstance(names_raw, dict)
                   else {i: str(v) for i, v in enumerate(names_raw)})
    for score, label, raw_mask in zip(scores, labels, masks):
        if float(score) < float(config["LOW_CONFIDENCE_FLOOR"]):
            continue
        if raw_mask.shape != (height, width):
            raw_mask = cv2.resize(raw_mask, (width, height), interpolation=cv2.INTER_NEAREST)
        binary_mask = raw_mask >= 0.5
        geometry = mask_geometry(binary_mask)
        if geometry is None:
            continue
        detections.append({
            "class_id": int(label),
            "class_name": class_names.get(int(label), f"class_{int(label)}"),
            "confidence": float(score),
            "mask": binary_mask,
            **geometry,
        })
    return detections, elapsed_seconds


def normalize_detections(detections, image_shape, confidence_floor=0.0):
    """Standardize predictor output to original-image binary masks and geometry fields."""
    height, width = int(image_shape[0]), int(image_shape[1])
    normalized = []
    for source in detections:
        confidence = float(source["confidence"])
        if confidence < float(confidence_floor):
            continue
        mask = np.asarray(source["mask"], dtype=bool)
        if mask.ndim != 2:
            raise ValueError("Each detection mask must be a 2-D binary array")
        if mask.shape != (height, width):
            mask = cv2.resize(mask.astype(np.uint8), (width, height),
                              interpolation=cv2.INTER_NEAREST).astype(bool)
        geometry = mask_geometry(mask)
        if geometry is None:
            continue
        detection = dict(source)
        detection.update(geometry)
        detection.update({
            "mask": mask,
            "confidence": confidence,
            "class_id": int(source.get("class_id", 0)),
            "class_name": str(source.get("class_name", "mushroom_cluster")),
        })
        normalized.append(detection)
    return normalized


def max_cardinality_matches(scores, valid):
    """Maximize valid match count first, then total IoU; unmatched rows use dummy columns."""
    scores = np.asarray(scores, dtype=np.float64)
    valid = np.asarray(valid, dtype=bool)
    n_tracks, n_detections = scores.shape
    if n_tracks == 0 or n_detections == 0:
        return []
    cardinality_bonus = float(min(n_tracks, n_detections) + 1)
    weights = np.zeros((n_tracks, n_detections + n_tracks), dtype=np.float64)
    weights[:, :n_detections] = np.where(valid, cardinality_bonus + scores, 0.0)
    row_indices, column_indices = linear_sum_assignment(-weights)
    return [(int(i), int(j), float(scores[i, j]))
            for i, j in zip(row_indices, column_indices)
            if j < n_detections and valid[i, j]]


def _pack_binary_mask(mask):
    mask = np.asarray(mask, dtype=bool)
    return np.packbits(mask.reshape(-1)), tuple(mask.shape)


def _packed_mask_iou(packed_a, shape_a, packed_b, shape_b):
    if tuple(shape_a) != tuple(shape_b):
        old_mask = np.unpackbits(packed_a, count=int(shape_a[0]) * int(shape_a[1])).reshape(shape_a)
        old_mask = cv2.resize(old_mask.astype(np.uint8), (shape_b[1], shape_b[0]),
                              interpolation=cv2.INTER_NEAREST).astype(bool)
        packed_a, shape_a = _pack_binary_mask(old_mask)
    intersection_bits = np.bitwise_and(packed_a, packed_b)
    if hasattr(np, "bitwise_count"):
        intersection = int(np.bitwise_count(intersection_bits).sum())
        area_a = int(np.bitwise_count(packed_a).sum())
        area_b = int(np.bitwise_count(packed_b).sum())
    else:
        area = int(shape_b[0]) * int(shape_b[1])
        a = np.unpackbits(packed_a, count=area)
        b = np.unpackbits(packed_b, count=area)
        intersection = int(np.count_nonzero(a & b))
        area_a = int(np.count_nonzero(a))
        area_b = int(np.count_nonzero(b))
    union = area_a + area_b - intersection
    return float(intersection / union) if union else 0.0


def binary_mask_iou(mask_a, mask_b):
    packed_a, shape_a = _pack_binary_mask(mask_a)
    packed_b, shape_b = _pack_binary_mask(mask_b)
    return _packed_mask_iou(packed_a, shape_a, packed_b, shape_b)


class MaskIoUTracker:
    """Sequence-local tracker: last observed binary mask, two confidence passes, no motion model."""
    def __init__(self, high_threshold=0.8710562586784363, low_floor=0.60,
                 association_iou=0.10, max_gap=0):
        self.high_threshold = float(high_threshold)
        self.low_floor = float(low_floor)
        self.association_iou = float(association_iou)
        self.max_gap = max_gap
        self.tracks = {}
        self.next_track_id = 1
        self.events = []
        if not (0 <= self.low_floor <= self.high_threshold <= 1):
            raise ValueError("Expected 0 <= low_floor <= high_threshold <= 1")
        if not (0 <= self.association_iou <= 1):
            raise ValueError("association_iou must be between 0 and 1")
        if self.max_gap is not None and int(self.max_gap) < 0:
            raise ValueError("max_gap must be >= 0 or None")

    def _associate(self, track_ids, detections):
        scores = np.zeros((len(track_ids), len(detections)), dtype=np.float64)
        valid = np.zeros_like(scores, dtype=bool)
        for i, track_id in enumerate(track_ids):
            state = self.tracks[track_id]
            old_packed, old_shape = state["last_mask_packed"], state["mask_shape"]
            for j, detection in enumerate(detections):
                iou = _packed_mask_iou(old_packed, old_shape,
                                       detection["_packed_mask"], detection["_mask_shape"])
                scores[i, j] = iou
                valid[i, j] = iou >= self.association_iou
        return max_cardinality_matches(scores, valid)

    def _update_track(self, track_id, detection, order, stage):
        state = self.tracks[track_id]
        state["last_mask_packed"] = detection["_packed_mask"]
        state["mask_shape"] = detection["_mask_shape"]
        state["last_order"] = int(order)
        state["observations"] += 1
        detection["track_id"] = int(track_id)
        detection["association_stage"] = stage

    def update(self, detections, observation_order):
        order = int(observation_order)
        for detection in detections:
            detection["track_id"] = None
            detection["association_stage"] = "untracked_low_confidence"
            detection["_packed_mask"], detection["_mask_shape"] = _pack_binary_mask(detection["mask"])

        eligible = sorted(
            track_id for track_id, state in self.tracks.items()
            if self.max_gap is None or order - state["last_order"] <= int(self.max_gap) + 1
        )
        high = [d for d in detections if float(d["confidence"]) >= self.high_threshold]
        low = [d for d in detections if self.low_floor <= float(d["confidence"]) < self.high_threshold]
        matched_tracks, used_high, used_low = set(), set(), set()

        # Pass 1: high-confidence detections can continue existing tracks.
        for track_index, detection_index, _ in self._associate(eligible, high):
            track_id = eligible[track_index]
            detection = high[detection_index]
            self._update_track(track_id, detection, order, "high_confidence")
            matched_tracks.add(track_id)
            used_high.add(detection_index)

        # Pass 2: low-confidence detections can recover unmatched tracks but cannot create tracks.
        remaining_tracks = [track_id for track_id in eligible if track_id not in matched_tracks]
        for track_index, detection_index, _ in self._associate(remaining_tracks, low):
            track_id = remaining_tracks[track_index]
            detection = low[detection_index]
            gap = order - self.tracks[track_id]["last_order"] - 1
            self._update_track(track_id, detection, order, "low_confidence_recovery")
            matched_tracks.add(track_id)
            used_low.add(detection_index)
            if gap > 0:
                self.events.append({"observation_order": order, "event": "reassociated_after_gap",
                                    "track_id": track_id, "gap_observations": int(gap)})

        # Remaining high-confidence detections start new sequence-scoped tracks.
        for index, detection in enumerate(high):
            if index in used_high:
                continue
            track_id = self.next_track_id
            self.next_track_id += 1
            detection["track_id"] = track_id
            detection["association_stage"] = "new_high_confidence"
            self.tracks[track_id] = {
                "last_mask_packed": detection["_packed_mask"],
                "mask_shape": detection["_mask_shape"],
                "last_order": order,
                "observations": 1,
                "first_order": order,
                "first_area_px": int(detection["area_px"]),
            }

        # Close tracks once the allowed number of missed sampled observations is exceeded.
        if self.max_gap is not None:
            for track_id in list(self.tracks):
                if track_id in matched_tracks:
                    continue
                gap = order - self.tracks[track_id]["last_order"]
                if gap > int(self.max_gap):
                    self.events.append({"observation_order": order, "event": "track_closed_after_gap",
                                        "track_id": track_id, "gap_observations": int(gap)})
                    del self.tracks[track_id]

        return detections

## 5. Measure cluster size and change

For each predicted mask, the notebook records area in pixels and the width and height of its bounding box. For each track, it calculates the area change between consecutive observed images. If a real capture interval is known, it also reports elapsed time and an area-change rate.

Optional calibration converts pixel measurements to approximate physical dimensions. Set both `REFERENCE_LENGTH_PX` and `REFERENCE_LENGTH_MM` using a reference in the **same image plane** as the mushrooms. This is a simple scale conversion; it does not correct perspective or depth differences.

In [ ]:
def build_growth_history(tracked_df, deadband_percent=5.0):
    columns = ["sequence_id", "sequence_name", "track_id", "from_observation_order", "to_observation_order",
               "from_timestamp", "to_timestamp", "time_basis", "elapsed_minutes", "from_area_px", "to_area_px",
               "area_change_px", "relative_area_change_percent", "growth_direction", "area_change_per_hour_px2"]
    rows = []
    if tracked_df.empty:
        return pd.DataFrame(columns=columns)
    for (sequence_id, track_id), group in tracked_df.groupby(["sequence_id", "track_id"], sort=False):
        group = group.sort_values("observation_order").to_dict("records")
        for previous, current in zip(group, group[1:]):
            area_a, area_b = float(previous["area_px"]), float(current["area_px"])
            area_change = area_b - area_a
            relative = (area_change / area_a * 100.0) if area_a > 0 else np.nan
            direction = ("increasing" if relative > deadband_percent else
                         "decreasing" if relative < -deadband_percent else "stable") if np.isfinite(relative) else "undefined"
            elapsed = None
            if pd.notna(previous.get("elapsed_minutes")) and pd.notna(current.get("elapsed_minutes")):
                elapsed = float(current["elapsed_minutes"]) - float(previous["elapsed_minutes"])
            rate = (area_change / (elapsed / 60.0)) if elapsed is not None and elapsed > 0 else np.nan
            rows.append({
                "sequence_id": sequence_id, "sequence_name": current["sequence_name"], "track_id": int(track_id),
                "from_observation_order": int(previous["observation_order"]),
                "to_observation_order": int(current["observation_order"]),
                "from_timestamp": previous.get("timestamp"), "to_timestamp": current.get("timestamp"),
                "time_basis": current.get("time_basis"), "elapsed_minutes": elapsed,
                "from_area_px": area_a, "to_area_px": area_b, "area_change_px": area_change,
                "relative_area_change_percent": relative, "growth_direction": direction,
                "area_change_per_hour_px2": rate,
            })
    return pd.DataFrame(rows, columns=columns)


def _stable_color(index):
    hue = int((int(index) * 0.61803398875 % 1.0) * 179)
    hsv = np.uint8([[[hue, 210, 255]]])
    return tuple(int(x) for x in cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0, 0])


def render_overlay(image_bgr, detections):
    canvas = image_bgr.copy()
    for index, detection in enumerate(detections, start=1):
        track_id = detection.get("track_id")
        color = _stable_color(track_id if track_id is not None else 100000 + index)
        mask = np.asarray(detection["mask"], dtype=bool)
        if mask.any():
            tint = np.empty_like(canvas)
            tint[:] = color
            canvas[mask] = cv2.addWeighted(canvas[mask], 0.55, tint[mask], 0.45, 0)
            contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(canvas, contours, -1, color, 2)
        x, y = int(detection["bbox_x"]), int(detection["bbox_y"])
        w, h = int(detection["bbox_width"]), int(detection["bbox_height"])
        cv2.rectangle(canvas, (x, y), (x + w, y + h), color, 2)
        label = (f"ID {track_id}" if track_id is not None else "untracked")
        label += f" {float(detection['confidence']):.2f}"
        cv2.putText(canvas, label, (max(0, x), max(18, y - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
    return canvas


def _append_csv_rows(path, rows):
    if not rows:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)


def _calibrated_geometry(row, config):
    px = config.get("REFERENCE_LENGTH_PX")
    mm = config.get("REFERENCE_LENGTH_MM")
    if px is None and mm is None:
        return row
    mm_per_px = float(mm) / float(px)
    row["width_mm"] = float(row["bbox_width"]) * mm_per_px
    row["height_mm"] = float(row["bbox_height"]) * mm_per_px
    row["area_mm2"] = float(row["area_px"]) * (mm_per_px ** 2)
    return row


def _plot_track_growth(tracked_df, plot_dir):
    if tracked_df.empty:
        return
    plot_dir = Path(plot_dir)
    for (sequence_id, track_id), group in tracked_df.groupby(["sequence_id", "track_id"], sort=False):
        group = group.sort_values("observation_order")
        timed = group["elapsed_minutes"].notna().all() and group["elapsed_minutes"].nunique() > 1
        x = group["elapsed_minutes"].to_numpy(dtype=float) / 60.0 if timed else group["observation_order"].to_numpy(dtype=int)
        xlabel = "Elapsed time (hours)" if timed else "Observation order"
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(x, group["area_px"].to_numpy(dtype=float), marker="o", linewidth=1.5)
        ax.set_title(f"{sequence_id} — track {int(track_id)}")
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Mask area (px²)")
        ax.grid(True, alpha=0.25)
        fig.tight_layout()
        target = plot_dir / str(sequence_id) / f"track_{int(track_id):04d}.png"
        target.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(target, dpi=140)
        plt.close(fig)


def build_diagnostics(detections_df, tracked_df, image_df, growth_df, jump_warning_percent=30.0):
    detections = detections_df.copy() if detections_df is not None else pd.DataFrame()
    tracked = tracked_df.copy() if tracked_df is not None else pd.DataFrame()
    images = image_df.copy() if image_df is not None else pd.DataFrame()
    growth = growth_df.copy() if growth_df is not None else pd.DataFrame()

    if not detections.empty:
        for col in ["confidence", "area_px", "bbox_width", "bbox_height", "image_width", "image_height"]:
            detections[col] = pd.to_numeric(detections[col], errors="coerce")
        detections["track_id_num"] = pd.to_numeric(detections["track_id"], errors="coerce")
        bbox_area = (detections["bbox_width"] * detections["bbox_height"]).replace(0, np.nan)
        image_area = (detections["image_width"] * detections["image_height"]).replace(0, np.nan)
        detections["mask_fill_pct"] = detections["area_px"] / bbox_area * 100.0
        detections["image_coverage_pct"] = detections["area_px"] / image_area * 100.0
        keys = ["sequence_id", "sequence_name", "image"]
        per_frame = detections.groupby(keys, dropna=False).agg(
            detection_count=("confidence", "size"),
            tracked_detection_count=("track_id_num", "count"),
            mean_confidence=("confidence", "mean"),
            min_confidence=("confidence", "min"),
            mean_mask_area_px=("area_px", "mean"),
            median_mask_area_px=("area_px", "median"),
            max_mask_area_px=("area_px", "max"),
            mean_bbox_width_px=("bbox_width", "mean"),
            mean_bbox_height_px=("bbox_height", "mean"),
            mean_mask_fill_pct=("mask_fill_pct", "mean"),
            summed_mask_area_px=("area_px", "sum"),
        ).reset_index()
        dims = detections[keys + ["image_width", "image_height"]].drop_duplicates(keys)
        per_frame = per_frame.merge(dims, on=keys, how="left")
        frame_pixels = (per_frame["image_width"] * per_frame["image_height"]).replace(0, np.nan)
        per_frame["summed_instance_coverage_pct"] = per_frame["summed_mask_area_px"] / frame_pixels * 100.0
        confidence = detections["confidence"].dropna()
        confidence_stats = {
            "count": int(len(confidence)), "mean": float(confidence.mean()),
            "median": float(confidence.median()), "p10": float(confidence.quantile(.10)),
            "p90": float(confidence.quantile(.90)), "min": float(confidence.min()),
            "max": float(confidence.max()),
        }
        bins = pd.cut(confidence, [-np.inf, .70, .80, .90, np.inf],
                      labels=["<0.70", "0.70–0.80", "0.80–0.90", "≥0.90"], right=False)
        confidence_bands = bins.value_counts(sort=False).rename_axis("confidence_band").reset_index(name="detections")
        stage_counts = detections["association_stage"].fillna("unknown").value_counts().rename_axis("association_stage").reset_index(name="count")
        geometry = {}
        for col in ["area_px", "bbox_width", "bbox_height", "mask_fill_pct", "image_coverage_pct"]:
            values = detections[col].replace([np.inf, -np.inf], np.nan).dropna()
            geometry[col] = {"mean": float(values.mean()), "median": float(values.median()),
                             "p90": float(values.quantile(.90)), "max": float(values.max())}
        tiny_masks = int((detections["area_px"] < 100).sum())
        untracked_detections = int(detections["track_id_num"].isna().sum())
    else:
        per_frame = pd.DataFrame(columns=["sequence_id", "sequence_name", "image", "detection_count", "tracked_detection_count"])
        confidence_stats = {"count": 0, "mean": None, "median": None, "p10": None, "p90": None, "min": None, "max": None}
        confidence_bands = pd.DataFrame(columns=["confidence_band", "detections"])
        stage_counts = pd.DataFrame(columns=["association_stage", "count"])
        geometry = {}
        tiny_masks = untracked_detections = 0

    if not images.empty:
        if "tracked_detections" not in images:
            images["tracked_detections"] = 0
        images["detections"] = pd.to_numeric(images["detections"], errors="coerce").fillna(0).astype(int)
        images["tracked_detections"] = pd.to_numeric(images["tracked_detections"], errors="coerce").fillna(0).astype(int)
        frame = images.merge(per_frame, on=["sequence_id", "sequence_name", "image"], how="left", suffixes=("", "_pred"))
        pred_count = pd.to_numeric(frame.get("detection_count", pd.Series(np.nan, index=frame.index)), errors="coerce")
        frame["detection_count"] = pred_count.fillna(frame["detections"]).fillna(0).astype(int)
        tracked_count = pd.to_numeric(frame.get("tracked_detection_count", pd.Series(np.nan, index=frame.index)), errors="coerce")
        frame["tracked_detection_count"] = tracked_count.fillna(frame["tracked_detections"]).fillna(0).astype(int)
        frame["zero_detection"] = (frame["status"] == "processed") & (frame["detection_count"] == 0)
    else:
        frame = per_frame.copy()
        if not frame.empty:
            frame["status"] = "processed"
            frame["zero_detection"] = frame["detection_count"] == 0

    track_rows = []
    if not tracked.empty:
        for (seq_id, seq_name, track_id), group in tracked.groupby(["sequence_id", "sequence_name", "track_id"], sort=False):
            group = group.sort_values("observation_order")
            first, last = group.iloc[0], group.iloc[-1]
            first_area, last_area = float(first["area_px"]), float(last["area_px"])
            net_pct = (last_area - first_area) / first_area * 100.0 if first_area > 0 else np.nan
            steps = growth[(growth["sequence_id"] == seq_id) & (growth["track_id"] == int(track_id))] if not growth.empty else pd.DataFrame()
            changes = pd.to_numeric(steps.get("relative_area_change_percent", pd.Series(dtype=float)), errors="coerce").dropna()
            track_rows.append({
                "sequence_id": seq_id, "sequence_name": seq_name, "track_id": int(track_id),
                "observations": int(len(group)), "first_order": int(first["observation_order"]),
                "last_order": int(last["observation_order"]), "first_image": first["image"],
                "last_image": last["image"], "first_area_px": first_area, "last_area_px": last_area,
                "net_area_change_pct": net_pct, "mean_confidence": float(group["confidence"].mean()),
                "min_confidence": float(group["confidence"].min()),
                "recovery_observations": int((group["association_stage"] == "low_confidence_recovery").sum()),
                "growth_transitions": int(len(changes)),
                "median_step_change_pct": float(changes.median()) if len(changes) else np.nan,
                "largest_abs_step_change_pct": float(changes.abs().max()) if len(changes) else np.nan,
            })
    tracks = pd.DataFrame(track_rows)

    if not growth.empty:
        growth["relative_area_change_percent"] = pd.to_numeric(growth["relative_area_change_percent"], errors="coerce")
        growth_flags = growth[growth["relative_area_change_percent"].abs() >= float(jump_warning_percent)].copy()
        values = growth["relative_area_change_percent"].dropna()
        growth_stats = {
            "transitions": int(len(growth)), "mean_change_pct": float(values.mean()) if len(values) else None,
            "median_change_pct": float(values.median()) if len(values) else None,
            "increasing": int((growth["growth_direction"] == "increasing").sum()),
            "stable": int((growth["growth_direction"] == "stable").sum()),
            "decreasing": int((growth["growth_direction"] == "decreasing").sum()),
            "large_change_flags": int(len(growth_flags)),
            "max_abs_change_pct": float(values.abs().max()) if len(values) else None,
        }
    else:
        growth_flags = pd.DataFrame(columns=growth.columns)
        growth_stats = {"transitions": 0, "mean_change_pct": None, "median_change_pct": None,
                        "increasing": 0, "stable": 0, "decreasing": 0,
                        "large_change_flags": 0, "max_abs_change_pct": None}

    return {"detections": detections, "frames": frame, "tracks": tracks, "growth": growth,
            "growth_flags": growth_flags, "confidence_stats": confidence_stats,
            "confidence_bands": confidence_bands, "stage_counts": stage_counts,
            "geometry": geometry, "growth_stats": growth_stats,
            "untracked_detections": untracked_detections, "tiny_masks": tiny_masks}


def print_diagnostics(diag, config):
    """Print a compact review of the predictions. These are not accuracy scores."""
    def show(value, digits=2):
        return "n/a" if value is None or pd.isna(value) else f"{float(value):.{digits}f}"
    frames, tracks, growth = diag["frames"], diag["tracks"], diag["growth"]
    conf, stats, geom = diag["confidence_stats"], diag["growth_stats"], diag["geometry"]
    print("\n" + "=" * 78 + "\nPREDICTION CHECK — no ground-truth accuracy is calculated\n" + "=" * 78)
    print("\nDETECTIONS")
    print(f"Found: {conf['count']:,} | confidence mean/median: {show(conf['mean'])}/{show(conf['median'])} | p10/p90: {show(conf['p10'])}/{show(conf['p90'])} | min/max: {show(conf['min'])}/{show(conf['max'])}")
    if not diag["confidence_bands"].empty:
        print(diag["confidence_bands"].to_string(index=False))
    if not frames.empty:
        done = frames[frames["status"] == "processed"] if "status" in frames else frames
        zero = int(done["zero_detection"].sum()) if "zero_detection" in done else 0
        print(f"Images processed: {len(done):,} | with detections: {len(done)-zero:,} | no detections: {zero:,}")
        if len(done): print(f"Detections/image: mean {show(done['detection_count'].mean())}, median {show(done['detection_count'].median())}, max {int(done['detection_count'].max())}")
    print("\nMASKS")
    if geom:
        for key, label, unit in [("area_px","Area","px²"),("bbox_width","Width","px"),("bbox_height","Height","px"),("mask_fill_pct","Mask fill in box","%")]:
            st=geom[key]; print(f"{label}: mean {show(st['mean'])} {unit} | median {show(st['median'])} | p90 {show(st['p90'])} | max {show(st['max'])}")
        print(f"Masks smaller than 100 px²: {diag['tiny_masks']}")
        print("Instance coverage sums overlapping masks if they overlap.")
    else: print("No masks were produced.")
    print("\nTRACKS")
    print(f"Tracks: {len(tracks):,} | tracked observations: {int(tracks['observations'].sum()) if not tracks.empty else 0:,} | untracked detections: {diag['untracked_detections']:,}")
    if not tracks.empty:
        print(f"Track length: mean {show(tracks['observations'].mean())}, median {show(tracks['observations'].median())}, longest {int(tracks['observations'].max())}")
        print("Recovery observations:", int(tracks["recovery_observations"].sum()))
        print("\nLongest tracks:")
        cols=[c for c in ["sequence_id","track_id","observations","first_area_px","last_area_px","net_area_change_pct","mean_confidence","recovery_observations","largest_abs_step_change_pct"] if c in tracks]
        print(tracks.sort_values("observations",ascending=False)[cols].head(int(config["CONSOLE_MAX_TRACK_ROWS"])).to_string(index=False, float_format=lambda x:f"{x:.2f}"))
    if not diag["stage_counts"].empty:
        print("\nAssociation stages:\n", diag["stage_counts"].to_string(index=False))
    print("\nGROWTH HISTORY")
    print(f"Transitions: {stats['transitions']:,} | increasing/stable/decreasing: {stats['increasing']}/{stats['stable']}/{stats['decreasing']}")
    print(f"Step change mean/median: {show(stats['mean_change_pct'])}%/{show(stats['median_change_pct'])}% | max absolute: {show(stats['max_abs_change_pct'])}%")
    print(f"Large changes flagged (>= {float(config['GROWTH_JUMP_WARNING_PERCENT']):g}%): {stats['large_change_flags']}")
    if not growth.empty:
        cols=[c for c in ["sequence_id","track_id","from_observation_order","to_observation_order","elapsed_minutes","from_area_px","to_area_px","relative_area_change_percent","growth_direction"] if c in growth]
        view=growth.assign(_abs=growth["relative_area_change_percent"].abs()).sort_values("_abs",ascending=False).head(int(config["CONSOLE_MAX_GROWTH_ROWS"]))
        print("\nLargest changes to inspect:\n", view[cols].to_string(index=False, float_format=lambda x:f"{x:.2f}"))
    if not frames.empty:
        cols=[c for c in ["sequence_id","observation_order","image","status","detection_count","tracked_detection_count","mean_confidence","mean_mask_area_px","summed_instance_coverage_pct"] if c in frames]
        view=frames.sort_values(["sequence_id","observation_order"])
        limit=int(config["CONSOLE_MAX_FRAME_ROWS"])
        if len(view)>limit:
            zero=view[view["zero_detection"]].head(limit//2)
            busy=view.sort_values("detection_count",ascending=False).head(limit-len(zero))
            view=pd.concat([zero,busy]).drop_duplicates(["sequence_id","image"]).head(limit)
        print("\nPer-image check (sample):\n", view[cols].to_string(index=False, float_format=lambda x:f"{x:.2f}"))
    print("\nCheck overlays for missed clusters, merged masks, background detections, or broken boundaries.")
    print("Check growth curves for sudden jumps, flat tracks, and ID switches.")


## 6. Reusable code

The next cells contain the image, tracking, measurement, and export functions used by the run cell. Normal use does not require editing them. Developers adapting the notebook for another model or backend can start with the predictor interface in the final section.

In [ ]:
def validate_monitoring_config(config):
    defaults = {
        "SEQUENCE_MODE": "auto", "ORDERING_MODE": "auto", "TIMESTAMP_REGEX": None,
        "TIMESTAMP_FORMAT": None, "FRAME_INTERVAL_MINUTES": None,
        "MAX_IMAGES_PER_SEQUENCE": None,
        "YOLO_IMAGE_SIZE": 1024, "YOLO_NMS_IOU": 0.70, "YOLO_MAX_DETECTIONS": 1000,
        "LOW_CONFIDENCE_FLOOR": 0.60, "HIGH_CONFIDENCE_THRESHOLD": 0.8710562586784363,
        "ASSOCIATION_IOU_THRESHOLD": 0.10, "MAX_GAP": 0,
        "GROWTH_DEADBAND_PERCENT": 5.0, "GROWTH_JUMP_WARNING_PERCENT": 30.0,
        "CONSOLE_MAX_FRAME_ROWS": 60, "CONSOLE_MAX_TRACK_ROWS": 50, "CONSOLE_MAX_GROWTH_ROWS": 80,
        "PREVIEW_MAX_IMAGES": 12, "PREVIEW_MAX_TRACK_PLOTS": 12, "DEVICE": "auto",
        "REFERENCE_LENGTH_PX": None, "REFERENCE_LENGTH_MM": None,
        "EXPORT_OVERLAYS": True, "EXPORT_BINARY_MASKS": True, "CREATE_ZIP": True,
    }
    cfg = {**defaults, **dict(config)}
    required = ["IMAGE_DIR", "OUTPUT_DIR"]
    missing = [key for key in required if not cfg.get(key)]
    if missing:
        raise ValueError(f"Missing required config values: {missing}")
    if float(cfg["LOW_CONFIDENCE_FLOOR"]) > float(cfg["HIGH_CONFIDENCE_THRESHOLD"]):
        raise ValueError("LOW_CONFIDENCE_FLOOR must be <= HIGH_CONFIDENCE_THRESHOLD so recovery detections are retained.")
    if cfg.get("FRAME_INTERVAL_MINUTES") is not None and float(cfg["FRAME_INTERVAL_MINUTES"]) <= 0:
        raise ValueError("FRAME_INTERVAL_MINUTES must be positive or None")
    if cfg.get("MAX_IMAGES_PER_SEQUENCE") is not None and int(cfg["MAX_IMAGES_PER_SEQUENCE"]) < 1:
        raise ValueError("MAX_IMAGES_PER_SEQUENCE must be a positive integer or None")
    px, mm = cfg.get("REFERENCE_LENGTH_PX"), cfg.get("REFERENCE_LENGTH_MM")
    if (px is None) != (mm is None):
        raise ValueError("Set both REFERENCE_LENGTH_PX and REFERENCE_LENGTH_MM, or leave both as None")
    if px is not None and (float(px) <= 0 or float(mm) <= 0):
        raise ValueError("Calibration reference lengths must be positive")
    return cfg


def run_monitoring(model, config, predictor=predict_yolo_image, progress_callback=None,
                   iterator_factory=None, model_metadata=None):
    """Run ordered inference, sequence-local tracking, growth calculations, and exports.

    Predictor contract: predictor(model, image_bgr, config, device) -> (detections, inference_seconds).
    Each detection must contain a 2-D binary `mask` and numeric `confidence`; class_id/name are optional.
    Masks are normalized to the original image size and geometry is calculated here.
    """
    cfg = validate_monitoring_config(config)
    model_metadata = dict(model_metadata or {})
    total_started = time.perf_counter()
    input_dir = Path(cfg["IMAGE_DIR"]).expanduser().resolve()
    output_parent = Path(cfg["OUTPUT_DIR"]).expanduser().resolve()
    output_parent.mkdir(parents=True, exist_ok=True)
    run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = output_parent / f"run_{run_stamp}"
    suffix = 1
    while run_dir.exists():
        run_dir = output_parent / f"run_{run_stamp}_{suffix:02d}"
        suffix += 1
    run_dir.mkdir(parents=True)

    masks_dir = run_dir / "masks"
    overlays_dir = run_dir / "overlays"
    growth_plots_dir = run_dir / "growth_curves"
    detections_csv = run_dir / "detections.csv"
    tracked_csv = run_dir / "tracked_observations.csv"
    image_csv = run_dir / "image_summary.csv"
    log_lines = []
    order_manifests = []
    all_tracking_events = []
    skipped_count = 0
    processed_count = 0
    detections_count = 0
    tracked_observation_count = 0
    inference_seconds = 0.0
    tracking_seconds = 0.0
    active_device = cfg.get("DEVICE", "auto")
    if str(active_device).lower() == "auto":
        active_device = 0 if torch.cuda.is_available() else "cpu"
    elif str(active_device).lower() == "cpu":
        active_device = "cpu"

    def log(message):
        log_lines.append(str(message))
        print(message)

    sequences = discover_sequences(input_dir, cfg.get("SEQUENCE_MODE", "auto"), output_parent)
    total_discovered = sum(len(sequence["files"]) for sequence in sequences)
    total_selected = 0
    log(f"Input: {input_dir}")
    log(f"Sequences: {len(sequences)} | Images available: {total_discovered:,}")

    for sequence in sequences:
        ordered, order_info = order_sequence(
            sequence["files"], cfg.get("ORDERING_MODE", "auto"),
            cfg.get("TIMESTAMP_REGEX"), cfg.get("TIMESTAMP_FORMAT"),
            cfg.get("FRAME_INTERVAL_MINUTES")
        )
        available_in_sequence = len(ordered)
        sequence_limit = cfg.get("MAX_IMAGES_PER_SEQUENCE")
        if sequence_limit is not None and len(ordered) > int(sequence_limit):
            ordered = ordered[:int(sequence_limit)]
            log(f"[SAMPLE] {sequence['sequence_id']}: selected first {len(ordered):,} of {available_in_sequence:,} chronological images")
        total_selected += len(ordered)
        for warning_text in order_info["warnings"]:
            log(f"[ORDERING] {sequence['sequence_id']}: {warning_text}")
        order_manifests.append({
            "sequence_id": sequence["sequence_id"], "sequence_name": sequence["sequence_name"],
            "available_images": available_in_sequence, "selected_images": len(ordered), **order_info,
        })
        tracker = MaskIoUTracker(
            high_threshold=cfg["HIGH_CONFIDENCE_THRESHOLD"],
            low_floor=cfg["LOW_CONFIDENCE_FLOOR"],
            association_iou=cfg["ASSOCIATION_IOU_THRESHOLD"],
            max_gap=cfg["MAX_GAP"],
        )
        log(f"\n[{sequence['sequence_id']}] {sequence['sequence_name']}: {len(ordered):,} images | time basis: {order_info['time_basis']}")
        det_buffer, tracked_buffer, image_buffer = [], [], []
        image_iterator = iterator_factory(ordered, desc=f"{sequence['sequence_id']} {sequence['sequence_name']}") if iterator_factory else ordered
        for item in image_iterator:
            path = item["path"]
            relative_path = path.relative_to(input_dir).as_posix()
            image = cv2.imread(str(path), cv2.IMREAD_COLOR)
            if image is None:
                skipped_count += 1
                reason = "unreadable_image"
                log(f"[SKIP] {relative_path}: OpenCV could not decode this file")
                image_buffer.append({
                    "sequence_id": sequence["sequence_id"], "sequence_name": sequence["sequence_name"],
                    "observation_order": item["observation_order"], "frame_index": item["frame_index"],
                    "source_frame_index": item["source_frame_index"], "timestamp": item["timestamp"],
                    "elapsed_minutes": item["elapsed_minutes"], "time_basis": order_info["time_basis"],
                    "image": relative_path, "status": "skipped", "skip_reason": reason,
                    "image_width": None, "image_height": None, "detections": 0,
                    "tracked_detections": 0, "mean_confidence": np.nan,
                    "inference_seconds": 0.0, "tracking_seconds": 0.0, "overlay_path": "",
                })
            else:
                height, width = image.shape[:2]
                try:
                    try:
                        detections, inference_elapsed = predictor(model, image, cfg, active_device)
                    except RuntimeError as exc:
                        oom_type = getattr(torch.cuda, "OutOfMemoryError", None)
                        is_oom = "out of memory" in str(exc).lower() or (oom_type is not None and isinstance(exc, oom_type))
                        if not is_oom or str(active_device).lower() == "cpu":
                            raise
                        warning_text = f"CUDA out of memory at {relative_path}; retrying on CPU and using CPU for remaining images."
                        log("[WARNING] " + warning_text)
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        active_device = "cpu"
                        detections, inference_elapsed = predictor(model, image, cfg, active_device)
                    detections = normalize_detections(
                        detections, image.shape[:2], confidence_floor=cfg["LOW_CONFIDENCE_FLOOR"]
                    )
                    inference_seconds += float(inference_elapsed)
                    track_started = time.perf_counter()
                    detections = tracker.update(detections, item["observation_order"])
                    frame_tracking_elapsed = time.perf_counter() - track_started
                    tracking_seconds += frame_tracking_elapsed
                    frame_tracked_count = sum(d["track_id"] is not None for d in detections)
                    tracked_observation_count += frame_tracked_count

                    overlay = render_overlay(image, detections) if cfg.get("EXPORT_OVERLAYS", True) else None
                    for det_index, detection in enumerate(detections, start=1):
                        det_row = {
                            "sequence_id": sequence["sequence_id"], "sequence_name": sequence["sequence_name"],
                            "observation_order": item["observation_order"], "frame_index": item["frame_index"],
                            "source_frame_index": item["source_frame_index"], "timestamp": item["timestamp"],
                            "elapsed_minutes": item["elapsed_minutes"], "time_basis": order_info["time_basis"],
                            "image": relative_path, "image_width": width, "image_height": height,
                            "detection_index": det_index, "class_id": detection["class_id"],
                            "class_name": detection["class_name"], "confidence": detection["confidence"],
                            "track_id": detection["track_id"], "association_stage": detection["association_stage"],
                            "area_px": detection["area_px"], "bbox_x": detection["bbox_x"],
                            "bbox_y": detection["bbox_y"], "bbox_width": detection["bbox_width"],
                            "bbox_height": detection["bbox_height"], "centroid_x_px": detection["centroid_x_px"],
                            "centroid_y_px": detection["centroid_y_px"],
                        }
                        _calibrated_geometry(det_row, cfg)
                        if cfg.get("EXPORT_BINARY_MASKS", True):
                            track_suffix = f"_track_{int(detection['track_id']):04d}" if detection["track_id"] is not None else "_untracked"
                            mask_path = masks_dir / sequence["sequence_id"] / f"obs_{item['observation_order']+1:06d}_det_{det_index:04d}{track_suffix}.png"
                            mask_path.parent.mkdir(parents=True, exist_ok=True)
                            if not cv2.imwrite(str(mask_path), detection["mask"].astype(np.uint8) * 255):
                                raise IOError(f"Could not write binary mask: {mask_path}")
                            det_row["mask_path"] = mask_path.relative_to(run_dir).as_posix()
                        else:
                            det_row["mask_path"] = ""
                        det_buffer.append(det_row)
                        if detection["track_id"] is not None:
                            tracked_buffer.append(det_row.copy())
                    if cfg.get("EXPORT_OVERLAYS", True):
                        overlay_path = overlays_dir / sequence["sequence_id"] / f"obs_{item['observation_order']+1:06d}.jpg"
                        overlay_path.parent.mkdir(parents=True, exist_ok=True)
                        if not cv2.imwrite(str(overlay_path), overlay):
                            raise IOError(f"Could not write overlay: {overlay_path}")
                        overlay_relative = overlay_path.relative_to(run_dir).as_posix()
                    else:
                        overlay_relative = ""
                    detections_count += len(detections)
                    processed_count += 1
                    image_buffer.append({
                        "sequence_id": sequence["sequence_id"], "sequence_name": sequence["sequence_name"],
                        "observation_order": item["observation_order"], "frame_index": item["frame_index"],
                        "source_frame_index": item["source_frame_index"], "timestamp": item["timestamp"],
                        "elapsed_minutes": item["elapsed_minutes"], "time_basis": order_info["time_basis"],
                        "image": relative_path, "status": "processed", "skip_reason": "",
                        "image_width": width, "image_height": height, "detections": len(detections),
                        "tracked_detections": sum(d["track_id"] is not None for d in detections),
                        "mean_confidence": float(np.mean([d["confidence"] for d in detections])) if detections else np.nan,
                        "inference_seconds": float(inference_elapsed),
                        "tracking_seconds": float(frame_tracking_elapsed), "overlay_path": overlay_relative,
                    })
                finally:
                    del image
                    if (int(item["observation_order"]) + 1) % 25 == 0:
                        gc.collect()
                        if torch.cuda.is_available() and str(active_device).lower() != "cpu":
                            torch.cuda.empty_cache()

            if len(det_buffer) >= 100:
                _append_csv_rows(detections_csv, det_buffer); det_buffer.clear()
            if len(tracked_buffer) >= 100:
                _append_csv_rows(tracked_csv, tracked_buffer); tracked_buffer.clear()
            if len(image_buffer) >= 50:
                _append_csv_rows(image_csv, image_buffer); image_buffer.clear()
            if progress_callback is not None:
                progress_callback({
                    "sequence_id": sequence["sequence_id"], "sequence_name": sequence["sequence_name"],
                    "observation_order": item["observation_order"], "image": relative_path,
                    "sequence_images": len(ordered),
                    "processed": processed_count, "skipped": skipped_count,
                    "detections": detections_count,
                    "tracked_observations": tracked_observation_count,
                })

        _append_csv_rows(detections_csv, det_buffer)
        _append_csv_rows(tracked_csv, tracked_buffer)
        _append_csv_rows(image_csv, image_buffer)
        for event in tracker.events:
            log(f"[TRACK] {sequence['sequence_id']} {event}")
            all_tracking_events.append({
                "sequence_id": sequence["sequence_id"],
                "sequence_name": sequence["sequence_name"],
                **event,
            })

    pd.DataFrame(all_tracking_events).to_csv(run_dir / "tracking_events.csv", index=False)
    detections_df = pd.read_csv(detections_csv) if detections_csv.exists() else pd.DataFrame()
    tracked_df = pd.read_csv(tracked_csv) if tracked_csv.exists() else pd.DataFrame()
    image_df = pd.read_csv(image_csv) if image_csv.exists() else pd.DataFrame()
    if not tracked_df.empty:
        tracked_df["track_id"] = tracked_df["track_id"].astype(int)
    growth_df = build_growth_history(tracked_df, float(cfg["GROWTH_DEADBAND_PERCENT"]))
    growth_df.to_csv(run_dir / "growth_history.csv", index=False)
    _plot_track_growth(tracked_df, growth_plots_dir)

    track_rows = []
    if not tracked_df.empty:
        for (sequence_id, sequence_name, track_id), group in tracked_df.groupby(
                ["sequence_id", "sequence_name", "track_id"], sort=False):
            group = group.sort_values("observation_order")
            first, last = group.iloc[0], group.iloc[-1]
            first_area, last_area = float(first.area_px), float(last.area_px)
            step_rows = growth_df[(growth_df["sequence_id"] == sequence_id) & (growth_df["track_id"] == int(track_id))] if not growth_df.empty else pd.DataFrame()
            step_values = pd.to_numeric(step_rows.get("relative_area_change_percent", pd.Series(dtype=float)), errors="coerce").dropna()
            net_pct = (last_area - first_area) / first_area * 100.0 if first_area > 0 else np.nan
            track_rows.append({
                "sequence_id": sequence_id, "sequence_name": sequence_name, "track_id": int(track_id),
                "observations": int(len(group)), "first_observation_order": int(first.observation_order),
                "last_observation_order": int(last.observation_order), "first_image": first.image,
                "last_image": last.image, "first_timestamp": first.timestamp,
                "last_timestamp": last.timestamp, "first_area_px": first_area,
                "last_area_px": last_area, "area_change_px": last_area - first_area,
                "net_area_change_pct": net_pct, "mean_confidence": float(group.confidence.mean()),
                "min_confidence": float(group.confidence.min()),
                "recovery_observations": int((group.association_stage == "low_confidence_recovery").sum()),
                "growth_transitions": int(len(step_values)),
                "median_step_change_pct": float(step_values.median()) if len(step_values) else np.nan,
                "largest_abs_step_change_pct": float(step_values.abs().max()) if len(step_values) else np.nan,
            })
    track_df = pd.DataFrame(track_rows)
    track_df.to_csv(run_dir / "track_summary.csv", index=False)

    diagnostics = build_diagnostics(
        detections_df, tracked_df, image_df, growth_df,
        jump_warning_percent=float(cfg.get("GROWTH_JUMP_WARNING_PERCENT", 30.0)),
    )
    diagnostics["frames"].to_csv(run_dir / "image_diagnostics.csv", index=False)
    diagnostics["tracks"].to_csv(run_dir / "track_diagnostics.csv", index=False)
    diagnostics["growth_flags"].to_csv(run_dir / "growth_review_flags.csv", index=False)

    wall_seconds = time.perf_counter() - total_started
    processed = int((image_df.status == "processed").sum()) if not image_df.empty else 0
    skipped = int((image_df.status == "skipped").sum()) if not image_df.empty else 0
    track_count = int(len(track_df))
    track_lengths = track_df["observations"].to_numpy(dtype=float) if track_count else np.array([])
    summary = {
        "input_directory": str(input_dir), "sequence_count": len(sequences),
        "images_discovered": int(total_discovered), "images_selected": int(total_selected), "images_processed": processed,
        "images_skipped": skipped, "detections": int(len(detections_df)),
        "images_with_detections": int((image_df.detections > 0).sum()) if not image_df.empty else 0,
        "tracked_observations": int(len(tracked_df)), "tracks": track_count,
        "untracked_detections": int(diagnostics["untracked_detections"]),
        "growth_transitions": int(diagnostics["growth_stats"]["transitions"]),
        "large_growth_change_flags": int(diagnostics["growth_stats"]["large_change_flags"]),
        "mean_track_length": float(track_lengths.mean()) if track_count else None,
        "median_track_length": float(np.median(track_lengths)) if track_count else None,
        "longest_track_length": int(track_lengths.max()) if track_count else 0,
        "inference_seconds": float(inference_seconds), "tracking_seconds": float(tracking_seconds),
        "total_wall_seconds": float(wall_seconds),
        "images_per_second": float(processed / wall_seconds) if wall_seconds > 0 else None,
        "device_used_at_end": str(active_device),
        "calibration_enabled": cfg.get("REFERENCE_LENGTH_PX") is not None,
        "time_bases": sorted({m["time_basis"] for m in order_manifests}),
    }
    try:
        torchvision_version = version("torchvision")
    except PackageNotFoundError:
        torchvision_version = None
    try:
        ultralytics_version = version("ultralytics")
    except PackageNotFoundError:
        ultralytics_version = None
    run_config = {
        "created_local": datetime.now().astimezone().isoformat(),
        "input_directory": str(input_dir), "output_directory": str(run_dir),
        "checkpoint": model_metadata.get("checkpoint", getattr(model, "ckpt_path", None)),
        "checkpoint_sha256": model_metadata.get("checkpoint_sha256"),
        "ultralytics": ultralytics_version, "torch": torch.__version__,
        "torchvision": torchvision_version, "numpy": np.__version__,
        "opencv": cv2.__version__, "scipy": scipy.__version__,
        "config": {k: (str(v) if isinstance(v, Path) else v) for k, v in cfg.items()},
        "sequence_ordering": order_manifests, "summary": summary,
    }
    (run_dir / "run_config.json").write_text(json.dumps(run_config, indent=2, default=str), encoding="utf-8")
    (run_dir / "run_log.txt").write_text("\n".join(log_lines), encoding="utf-8")

    summary_lines = [
        "OYSTER MUSHROOM CLUSTER MONITORING SUMMARY",
        "=" * 48,
        f"Input: {input_dir}",
        f"Sequences: {len(sequences)}",
        f"Images: {processed:,} processed / {total_selected:,} selected / {total_discovered:,} available / {skipped:,} skipped",
        f"Detections: {len(detections_df):,} | images with detections: {summary['images_with_detections']:,}",
        f"Tracked observations: {len(tracked_df):,} | tracks: {track_count:,}",
        f"Track length (mean / median / max): {summary['mean_track_length']} / {summary['median_track_length']} / {summary['longest_track_length']}",
        f"Inference time: {inference_seconds:.2f} s | tracker time: {tracking_seconds:.2f} s | wall time: {wall_seconds:.2f} s",
        f"Throughput: {summary['images_per_second']:.2f} images/s" if summary['images_per_second'] is not None else "Throughput: n/a",
        f"Time basis: {', '.join(summary['time_bases'])}",
        f"Physical calibration: {'enabled' if summary['calibration_enabled'] else 'not set (pixel units only)'}",
        f"Output: {run_dir}",
    ]
    (run_dir / "console_summary.txt").write_text("\n".join(summary_lines), encoding="utf-8")
    print("\n" + "\n".join(summary_lines))
    import contextlib, io
    report_buffer = io.StringIO()
    with contextlib.redirect_stdout(report_buffer):
        print_diagnostics(diagnostics, cfg)
    inspection_report = report_buffer.getvalue()
    print(inspection_report)
    (run_dir / "inspection_report.txt").write_text(inspection_report, encoding="utf-8")

    zip_path = None
    if cfg.get("CREATE_ZIP", True):
        zip_path = run_dir.with_suffix(".zip")
        shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=run_dir)
        print("ZIP:", zip_path)

    return {
        "output_dir": run_dir, "zip_path": zip_path, "summary": summary,
        "detections": detections_df, "tracked_observations": tracked_df,
        "growth_history": growth_df, "track_summary": track_df,
        "image_summary": image_df, "diagnostics": diagnostics,
    }

## 7. Run the full dataset

This cell runs inference and tracking over all images found in `IMAGE_DIR`, then prints the inspection report and saves the results. To run a smaller test, change `MAX_IMAGES_PER_SEQUENCE` in the **top config cell**; do not edit this cell.

In [ ]:
CONFIG = {
    "IMAGE_DIR": IMAGE_DIR, "OUTPUT_DIR": OUTPUT_DIR,
    "SEQUENCE_MODE": SEQUENCE_MODE, "ORDERING_MODE": ORDERING_MODE,
    "TIMESTAMP_REGEX": TIMESTAMP_REGEX, "TIMESTAMP_FORMAT": TIMESTAMP_FORMAT,
    "FRAME_INTERVAL_MINUTES": FRAME_INTERVAL_MINUTES,
    "MAX_IMAGES_PER_SEQUENCE": MAX_IMAGES_PER_SEQUENCE,
    "YOLO_IMAGE_SIZE": YOLO_IMAGE_SIZE, "YOLO_NMS_IOU": YOLO_NMS_IOU,
    "YOLO_MAX_DETECTIONS": YOLO_MAX_DETECTIONS, "LOW_CONFIDENCE_FLOOR": LOW_CONFIDENCE_FLOOR,
    "HIGH_CONFIDENCE_THRESHOLD": HIGH_CONFIDENCE_THRESHOLD,
    "ASSOCIATION_IOU_THRESHOLD": ASSOCIATION_IOU_THRESHOLD, "MAX_GAP": MAX_GAP,
    "GROWTH_DEADBAND_PERCENT": GROWTH_DEADBAND_PERCENT,
    "GROWTH_JUMP_WARNING_PERCENT": GROWTH_JUMP_WARNING_PERCENT,
    "CONSOLE_MAX_FRAME_ROWS": CONSOLE_MAX_FRAME_ROWS,
    "CONSOLE_MAX_TRACK_ROWS": CONSOLE_MAX_TRACK_ROWS,
    "CONSOLE_MAX_GROWTH_ROWS": CONSOLE_MAX_GROWTH_ROWS,
    "PREVIEW_MAX_IMAGES": PREVIEW_MAX_IMAGES,
    "PREVIEW_MAX_TRACK_PLOTS": PREVIEW_MAX_TRACK_PLOTS,
    "DEVICE": DEVICE, "REFERENCE_LENGTH_PX": REFERENCE_LENGTH_PX,
    "REFERENCE_LENGTH_MM": REFERENCE_LENGTH_MM,
    "EXPORT_OVERLAYS": EXPORT_OVERLAYS, "EXPORT_BINARY_MASKS": EXPORT_BINARY_MASKS,
    "CREATE_ZIP": CREATE_ZIP,
}


def console_progress(state):
    position = int(state.get("observation_order", 0)) + 1
    total = int(state.get("sequence_images", position))
    if position % 25 == 0 or position == total:
        tqdm.write(
            f"[{state.get('sequence_name', state.get('sequence_id'))}] {position}/{total} | "
            f"processed {state.get('processed', 0)} | detections {state.get('detections', 0)} | "
            f"tracked {state.get('tracked_observations', 0)}"
        )


print("Starting inference...")
RESULT = run_monitoring(
    yolo_model, CONFIG,
    progress_callback=console_progress,
    iterator_factory=lambda items, desc: tqdm(items, desc=desc, unit="image"),
    model_metadata={"checkpoint": str(YOLO_CHECKPOINT), "checkpoint_sha256": YOLO_CHECKPOINT_SHA256},
)

## 8. Review the results

The next cell shows a small selection of original images beside their predicted mask overlays, plus growth curves for selected tracks. The preview limits in the config cell only control how many examples are displayed; they do **not** limit inference or the saved results.

The full run is saved under `OUTPUT_DIR` in a timestamped folder. Main files include:

| File | Contents |
|---|---|
| `detections.csv` | Every YOLO detection, including detections not assigned to a track. |
| `tracked_observations.csv` | Detections assigned to a track. |
| `growth_history.csv` | Area change between consecutive observations of each track. |
| `track_summary.csv` | Start/end observations, track length, and net area change. |
| `image_summary.csv` | Detection counts and processing status for each image. |
| `image_diagnostics.csv` | Per-image mask and detection checks. |
| `track_diagnostics.csv` | Per-track confidence, length, recovery, and growth summaries. |
| `growth_review_flags.csv` | Large one-step area changes worth checking in the overlays. |
| `inspection_report.txt` | The detailed console report saved as a text file. |
| `run_config.json` | The settings and model checkpoint used for the run. |

The `overlays/`, `masks/`, and `growth_curves/` folders contain the visual outputs. The notebook also creates a ZIP of the complete run when `CREATE_ZIP = True`.

Large area changes are flagged for manual review, not automatically treated as mistakes. Check the corresponding masks before interpreting a growth curve. Without annotations, the report describes model predictions; it does not calculate segmentation or tracking accuracy.

In [ ]:
from pathlib import Path
from IPython.display import display

print("TRACK SUMMARY")
display(RESULT["diagnostics"]["tracks"].sort_values("observations", ascending=False).head(CONSOLE_MAX_TRACK_ROWS))
print("GROWTH HISTORY — largest changes first")
if not RESULT["growth_history"].empty:
    growth_view = RESULT["growth_history"].assign(
        abs_change=RESULT["growth_history"]["relative_area_change_percent"].abs()
    ).sort_values("abs_change", ascending=False).head(CONSOLE_MAX_GROWTH_ROWS)
    display(growth_view.drop(columns="abs_change"))
else:
    print("No consecutive observations within tracks.")

# Show a few frames per sequence: first empty frame, most detections, middle, and last.
frames = RESULT["diagnostics"]["frames"].copy()
selected = []
if not frames.empty:
    for sequence_id, group in frames.groupby("sequence_id", sort=False):
        group = group.sort_values("observation_order")
        picks = []
        if "zero_detection" in group:
            empty = group[group["zero_detection"]]
            if not empty.empty: picks.append(empty.iloc[0])
        if "detection_count" in group:
            picks.append(group.sort_values("detection_count", ascending=False).iloc[0])
        picks.extend([group.iloc[len(group)//2], group.iloc[-1]])
        for row in picks:
            key = (row["sequence_id"], row["image"])
            if all((x["sequence_id"], x["image"]) != key for x in selected):
                selected.append(row)
selected = selected[:PREVIEW_MAX_IMAGES]

print(f"MASK REVIEW — showing {len(selected)} of {len(frames)} processed frames")
for row in selected:
    original_path = Path(RESULT["summary"]["input_directory"]) / str(row["image"])
    overlay_rel = row.get("overlay_path", "")
    overlay_path = Path(RESULT["output_dir"]) / str(overlay_rel) if isinstance(overlay_rel, str) and overlay_rel else None
    original = cv2.imread(str(original_path))
    overlay = cv2.imread(str(overlay_path)) if overlay_path and overlay_path.is_file() else None
    if original is None: continue
    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots(1, 2 if overlay is not None else 1, figsize=(14, 5))
    if not isinstance(axes, np.ndarray): axes = [axes]
    axes[0].imshow(original); axes[0].set_title("Original"); axes[0].axis("off")
    if overlay is not None:
        axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axes[1].set_title("YOLO masks and track IDs"); axes[1].axis("off")
    fig.suptitle(f"{row['sequence_id']} | {row['image']} | detections: {int(row.get('detection_count', 0))}")
    fig.tight_layout(); plt.show()

print("GROWTH CURVES")
track_table = RESULT["diagnostics"]["tracks"]
shown = 0
if not track_table.empty:
    for row in track_table.sort_values("observations", ascending=False).head(PREVIEW_MAX_TRACK_PLOTS).itertuples(index=False):
        curve_path = Path(RESULT["output_dir"]) / "growth_curves" / str(row.sequence_id) / f"track_{int(row.track_id):04d}.png"
        if curve_path.is_file():
            curve = cv2.imread(str(curve_path))
            plt.figure(figsize=(8, 3.5))
            plt.imshow(cv2.cvtColor(curve, cv2.COLOR_BGR2RGB))
            plt.title(f"{row.sequence_id} — track {int(row.track_id)} ({int(row.observations)} observations)")
            plt.axis("off"); plt.show(); shown += 1
if not shown: print("No growth curves available yet.")

print("\nOutput folder:", RESULT["output_dir"])
print("ZIP:", RESULT["zip_path"])

## Using the tracker with another model or backend

The main entry point is `run_monitoring(model, config, predictor=...)`. The default `predict_yolo_image()` function is the Ultralytics-specific part. To use another model or call a model service, provide your own predictor and keep the sequence, tracking, measurement, and export code.

The predictor must accept `(model, image_bgr, config, device)` and return:

```python
(detections, inference_seconds)
```

Each item in `detections` needs:

- `mask`: a 2-D binary mask for one cluster. It can be at the original image size or be resizeable to it.
- `confidence`: a numeric confidence score.

`class_id` and `class_name` are optional. The pipeline calculates mask area, bounding box, and centroid itself. To integrate this into an application, move the reusable functions/classes into a Python module and keep only the model-loading and predictor code specific to that application.